# Session 19: Grouping and Summarising

Data Science with Python, Level 1, Stage 2: NumPy and pandas

## Learning Objectives

By the end of this session you will be able to:

1. Explain split, apply, combine.
2. Summarise groups with `groupby`.
3. Calculate several statistics with `agg`.
4. Group by more than one column.
5. Build pivot tables and cross tables.
6. Add group results back to every row.

## Where We Are

In Session 17 you compared the three branches with a loop: filter one branch, calculate, repeat. That works for three branches, but not for 40 products or 365 days, and the results are scattered across printed lines instead of one table.

**Grouping** is the most used idea in data analysis. You saw its seed in Session 07, when you added up sales per category in a dictionary. pandas does it for a whole table in one line with `groupby`.

## Setup

Run this cell first. It loads the libraries and sets the address of the course datasets.

In [ ]:
import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
df = pd.read_csv(DATA_URL + "cafe_sales.csv")
df["revenue"] = df["unit_price"] * df["quantity"]
df["date"] = pd.to_datetime(df["date"])

## 1. Split, Apply, Combine

In [ ]:
df.groupby("branch")["revenue"].sum().round(2)

In [ ]:
print(df.groupby("category")["revenue"].mean().round(2))
print()
print(df.groupby("branch")["order_id"].count())

## 2. Several Statistics with agg

In [ ]:
summary = df.groupby("branch").agg(
    orders=("order_id", "count"),
    revenue=("revenue", "sum"),
    avg_order=("revenue", "mean"),
    items=("quantity", "sum"),
)
summary.round(2).sort_values("revenue", ascending=False)

| Name | Meaning |
| --- | --- |
| `"count"`, `"size"` | Number of values, or of rows |
| `"sum"`, `"mean"`, `"median"` | Total, average, middle value |
| `"min"`, `"max"` | Smallest and largest |
| `"std"` | Standard deviation (spread) |
| `"nunique"` | Number of different values |
| `"first"`, `"last"` | First or last value in each group |

## 3. Grouping by Several Columns

In [ ]:
by_two = df.groupby(["branch", "category"])["revenue"].sum().round(1)
print(by_two.head(7))

In [ ]:
table = by_two.unstack()
table

In [ ]:
monthly = df.groupby(df["date"].dt.month)["revenue"].sum().round(2)
print(monthly)

hourly = df.groupby("hour")["order_id"].count()
print("Busiest hour:", hourly.idxmax(), "with", hourly.max(), "orders")

## 4. Pivot Tables and Cross Tables

In [ ]:
pd.pivot_table(df, index="category", columns="branch",
               values="revenue", aggfunc="sum", margins=True).round(1)

In [ ]:
pd.crosstab(df["branch"], df["payment"], normalize="index").round(2)

## 5. Group Results on Every Row

`agg` returns one row per group. Sometimes you want the group result next to **every original row**, for example to compare each order with its branch average. `transform` does exactly that: it returns a Series as long as the original table.

In [ ]:
df["branch_avg"] = df.groupby("branch")["revenue"].transform("mean")
df["above_avg"] = df["revenue"] > df["branch_avg"]

print(df[["branch", "revenue", "branch_avg", "above_avg"]].head(4).round(2))
print(df.groupby("branch")["above_avg"].mean().round(2))

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| KeyError | Grouping by a column name with a typo | Check `df.columns` for the exact name. |
| Index confusion | Expecting the group names as a column | Call `.reset_index()` to turn them back into a column. |
| Wrong length | Using `agg` when you needed a value per row | Use `transform` to keep the original number of rows. |
| TypeError | Averaging a text column inside groups | Select numeric columns before the statistic. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: Revenue per Item

Find the total revenue of each item and show the top five, highest first.

In [ ]:
# Write your code below

### In-Class Exercise 2: Branch Summary

Build one summary table per branch with the number of orders, the average quantity, the number of different items sold and the total revenue.

In [ ]:
# Write your code below

### In-Class Exercise 3: Category by Hour

Make a pivot table with hours as rows, categories as columns and the number of orders as values (use `aggfunc="count"` and `values="order_id"`). When do sandwiches sell?

In [ ]:
# Write your code below

### In-Class Exercise 4: Share of Branch

Add a column with each order's share of its branch's total revenue, using `transform("sum")`. Check that the shares add up to 1 in each branch.

In [ ]:
# Write your code below

## Summary

- `groupby` splits, applies and combines in one line.
- `df.groupby("a")["b"].sum()` totals `b` per group of `a`.
- `agg` builds a table of several statistics.
- Group by a list of columns for nested groups; `unstack` widens.
- `pivot_table` and `crosstab` make spreadsheet-style summaries.
- `transform` puts group results on every row.

## What's Next

Real data is spread over several tables: customers in one file, orders in another, products in a third. Next session you will join them together with `merge` and stack them with `concat`.